# 02 — Baseline U-Net

Train the vanilla U-Net baseline and visualise predictions. For full runs prefer
the CLI (`python -m src.train`); this notebook is for interactive inspection.

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath('..'))
from src.utils import load_config
from src.train import main as train_main
# Quick smoke run: 2 epochs. Edit configs/config.yaml for real training.
# In a terminal:  python -m src.train --epochs 2
cfg = load_config('../configs/config.yaml'); cfg['model']['arch']

In [ ]:
# Visualise predictions from a trained checkpoint
import cv2, numpy as np, matplotlib.pyplot as plt
from src.predict import SegmentationPredictor
from src.utils import resolve
import pandas as pd

predictor = SegmentationPredictor('../checkpoints/best_unet.pt', '../configs/config.yaml')
df = pd.read_csv(resolve(cfg['data']['splits_dir']) / 'test.csv')

fig, axes = plt.subplots(3, 3, figsize=(11, 11))
for ax_row, (_, row) in zip(axes, df.sample(3, random_state=1).iterrows()):
    img = cv2.cvtColor(cv2.imread(str(resolve(row['image_path']))), cv2.COLOR_BGR2RGB)
    gt = cv2.imread(str(resolve(row['mask_path'])), cv2.IMREAD_GRAYSCALE)
    overlay, mask, area = predictor.predict_overlay(img)
    ax_row[0].imshow(img); ax_row[0].set_title('image'); ax_row[0].axis('off')
    ax_row[1].imshow(gt, cmap='gray'); ax_row[1].set_title('ground truth'); ax_row[1].axis('off')
    ax_row[2].imshow(overlay); ax_row[2].set_title(f'pred ({area*100:.1f}%)'); ax_row[2].axis('off')
plt.tight_layout(); plt.show()